# 03 — Offline judge for generated briefs

Score briefs in one `{run_id}/` folder with the same judge as in-app paper-brief evaluation. Write `03-evaluations.jsonl` next to `02-briefs.jsonl`.

**Prerequisite:** notebook 02 has written `{run_id}/02-briefs.jsonl`, and sibling `corpus/` has the matching `.txt` files. Set **MODEL** (required chat model id) before you run the judge cell. Empty or whitespace → stop; no scores written. `RUN_ID` still picks which step 02 folder to score; it is independent of the judge model (you can generate with `llama3.1:8b` and judge with `gemma4:e4b`). This notebook does **not** query Postgres and does **not** read or write `PaperBrief`. Run it with `just notebooks` (needs `OPENAI_*`). Do not use `just sandbox`.

Domain calls: `judge_paper_brief_evaluation` from `paper_reviewer.topic_scope.paper_brief_evaluation.llm`, and `mean_evaluation_score` from `paper_reviewer.schemas.topic_scope.paper_brief_evaluation`. Do not import `paper_reviewer.flows` and do not call `evaluate_paper_brief`.

**Git:** `{run_id}/` results under `data/paper_brief_evaluation/` are tracked so you can commit them. They stay out of the production image (`.dockerignore`).

In [ ]:
# Folder name under data/paper_brief_evaluation/ (example "20260818T160000Z_llama3.1-8b").
# Leave empty to use the latest run that already has 02-briefs.jsonl.
RUN_ID = ""

In [ ]:
# Chat model id for this judge run (required). Example: "llama3.1:8b" or "gemma4:e4b"
MODEL = "llama3.1:8b"

In [ ]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path

from paper_reviewer.schemas.topic_scope.generate_paper_brief import PaperBriefContent
from paper_reviewer.schemas.topic_scope.paper_brief_evaluation import (
    PaperBriefEvaluation,
    mean_evaluation_score,
)
from paper_reviewer.topic_scope.paper_brief_evaluation.llm import (
    judge_paper_brief_evaluation,
)


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "paper_reviewer").is_dir() and (
            candidate / "pyproject.toml"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Cannot find the repo root. Start Jupyter with `just notebooks` "
        "so the kernel can see /workspace."
    )


REPO_ROOT = repo_root()
RUNS_PARENT = REPO_ROOT / "data" / "paper_brief_evaluation"
CORPUS_DIR = RUNS_PARENT / "corpus"
print(f"repo root: {REPO_ROOT}")
print(f"corpus dir: {CORPUS_DIR}")
print(f"runs parent: {RUNS_PARENT}")

In [ ]:
_RUN_ID_PATTERN = re.compile(r"^\d{8}T\d{6}Z_.+$")


def corpus_filename(doi: str) -> str:
    return f"{doi.replace('/', '_')}.txt"


def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            stripped = line.strip()
            if stripped:
                rows.append(json.loads(stripped))
    return rows


def has_brief(row: dict) -> bool:
    return row.get("brief") is not None


def list_run_ids(parent: Path) -> list[str]:
    ids: list[str] = []
    if not parent.is_dir():
        return ids
    for child in parent.iterdir():
        if (
            child.is_dir()
            and _RUN_ID_PATTERN.fullmatch(child.name)
            and (child / "02-briefs.jsonl").is_file()
        ):
            ids.append(child.name)
    return sorted(ids)


def resolve_run_dir(parent: Path, run_id: str) -> Path:
    chosen = run_id.strip()
    if chosen:
        run_dir = parent / chosen
        briefs_path = run_dir / "02-briefs.jsonl"
        if not briefs_path.is_file():
            raise RuntimeError(
                f"Missing {briefs_path}. Run notebook 02 (generate briefs) first."
            )
        return run_dir
    ids = list_run_ids(parent)
    if not ids:
        raise RuntimeError(
            f"No run folder with 02-briefs.jsonl under {parent}. "
            "Run notebook 02 (generate briefs) first."
        )
    return parent / ids[-1]


def evaluation_success_record(
    doi: str, evaluation: PaperBriefEvaluation
) -> dict:
    return {
        "doi": doi,
        "evaluation_score": float(mean_evaluation_score(evaluation)),
        "evaluation": evaluation.model_dump(mode="json"),
    }


def evaluation_error_record(doi: str, error: str) -> dict:
    return {"doi": doi, "error": error}

In [ ]:
model = MODEL.strip()
if not model:
    raise RuntimeError(
        "MODEL is required. Set the chat model id in the MODEL cell "
        '(example: "llama3.1:8b"). No evaluations were written.'
    )
os.environ["OPENAI_MODEL"] = model

run_dir = resolve_run_dir(RUNS_PARENT, RUN_ID)
briefs_path = run_dir / "02-briefs.jsonl"
evaluations_path = run_dir / "03-evaluations.jsonl"
judge_model_path = run_dir / "03-judge-model.txt"
brief_rows = load_jsonl(briefs_path)
judge_model_path.write_text(model, encoding="utf-8")
evaluations_path.write_text("", encoding="utf-8")
print(f"model: {model}")
print(f"run dir: {run_dir}")
print(f"briefs: {briefs_path}")
print(f"brief rows: {len(brief_rows)}")

accepted = 0
skipped = 0
errors: list[tuple[str, str]] = []

for row in brief_rows:
    doi = str(row.get("doi") or "(missing doi)")
    if not has_brief(row):
        print(f"SKIP {doi}: no brief (step 2 error; no judge call)")
        skipped += 1
        continue
    try:
        content = PaperBriefContent.model_validate(row["brief"])
        text_path = CORPUS_DIR / corpus_filename(doi)
        if not text_path.is_file():
            raise FileNotFoundError(f"missing corpus file: {text_path}")
        full_text = text_path.read_text(encoding="utf-8")
        evaluation = judge_paper_brief_evaluation(full_text, content=content)
        record = evaluation_success_record(doi, evaluation)
        accepted += 1
        print(f"OK {doi} score={record['evaluation_score']}")
    except Exception as exc:
        message = str(exc)
        record = evaluation_error_record(doi, message)
        errors.append((doi, message))
        print(f"ERROR {doi}: {exc}")
    with evaluations_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

print("---")
print(f"accepted: {accepted}")
print(f"skipped (no brief): {skipped}")
print(f"errors: {len(errors)}")
print(f"evaluations: {evaluations_path}")

After a successful run, commit the `{run_id}/` folder if you want the scores in the repository:

```bash
git add data/paper_brief_evaluation/
```

Production images still exclude `data/` via `.dockerignore`.